# PGD-HJ for LASSO

Compares proximal-gradient descent with the analytical soft-thresholding proximal against the same algorithm using HJ-Prox, plus a proximal-point variant.  Reproduces panels of Figures 1 and 3 in the paper.

## Setup


In [ ]:
# ============================================================================
# CHUNK 1: SETUP - Algorithms, Helper Functions, and Definitions
# ============================================================================

import time
import torch
import numpy as np
from hj_prox import hj_prox
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

device = 'cpu'
EPS = 1e-5


# ============================================================================
# Helper Functions
# ============================================================================

def lasso_objective(x, A, b, lambda_1):
    """
    Compute LASSO objective: 0.5 * ||Ax - b||_2^2 + λ * ||x||_1
    
    Args:
        x: signal (n, 1) or batch (batch_size, n)
        A: measurement matrix (m, n)
        b: observations (m, 1)
        lambda_1: regularization parameter
    """
    if x.dim() == 1:
        x = x.unsqueeze(0)
    elif x.dim() == 2 and x.shape[0] != 1:
        x = x.t()
    
    residual = A @ x.t() - b
    data_fid = 0.5 * torch.norm(residual, p=2) ** 2
    penalty = lambda_1 * torch.norm(x, p=1)
    
    return data_fid + penalty


def compute_gradient(x, A, b):
    """Gradient of the smooth part: ∇f(x) = A^T(Ax - b)"""
    return A.t() @ (A @ x - b)


def soft_threshold(x, threshold):
    """
    Soft-thresholding operator (proximal operator for L1 norm)
    
    prox_{λ||·||_1}(x) = sign(x) * max(|x| - λ, 0)
    
    Args:
        x: input tensor
        threshold: threshold parameter λ
    
    Returns:
        Soft-thresholded tensor
    """
    return torch.sign(x) * torch.maximum(torch.abs(x) - threshold, torch.zeros_like(x))


# ============================================================================
# Algorithm 1: Proximal Gradient Descent (Analytical Soft-Thresholding)
# ============================================================================

def proximal_gradient_descent_analytical(
    x0, A, b, lambda_1, step_size, max_iters=1000, tol=1e-6, verbose=True
):
    """
    Proximal gradient descent for solving the LASSO problem using analytical soft-thresholding:
      minimize 0.5 * ||Ax - b||^2 + lambda * ||x||_1
    
    This implementation uses the closed-form soft-thresholding operator.
    """
    xk = x0.clone()
    f_hist = []
    diff_hist = []
    
    for i in range(max_iters):
        t0 = time.time()
        
        # Gradient step
        grad = compute_gradient(xk, A, b)
        x_grad = xk - step_size * grad
        
        # Proximal step using soft-thresholding
        x_prox = soft_threshold(x_grad, step_size * lambda_1)
        
        # Compute metrics
        fk = lasso_objective(x_prox, A, b, lambda_1)
        diff = torch.norm(x_prox - xk)
        
        f_hist.append(fk.item())
        diff_hist.append(diff.item())
        xk = x_prox.clone()
        
        if verbose and i % 10 == 0:
            print(f"Analytical PGD iter {i+1:4d}: f={fk.item():.6f}, "
                  f"||Δx||={diff.item():.6e}, time={time.time() - t0:.4f}s")
    
    return xk, torch.tensor(f_hist), torch.tensor(diff_hist)


# ============================================================================
# Algorithm 2: Proximal Gradient Descent with HJ-Prox
# ============================================================================

def proximal_gradient_descent_lasso(
    x0, A, b, lambda_1, step_size, max_iters=1000, 
    num_samples=100, delta=None, tol=1e-6, verbose=True
):
    """
    Proximal gradient descent for solving the LASSO problem:
      minimize 0.5 * ||Ax - b||^2 + lambda * ||x||_1
    
    Uses HJ-Prox with delta = 250000/(k+1)^(2+EPS) annealing schedule.
    """
    xk = x0.clone()
    f_hist = []
    diff_hist = []
    
    # Define L1 penalty function for HJ-Prox
    def l1_penalty(x_batch):
        if x_batch.dim() == 1:
            x_batch = x_batch.unsqueeze(0)
        return lambda_1 * torch.sum(torch.abs(x_batch), dim=1)
    
    for i in range(max_iters):
        t0 = time.time()
        
        # Compute delta with annealing schedule
        k = i + 1
        delta_k = 125000/(i+1)**(2+EPS)
        t_k = 1/(i+1)**(1+EPS)
        
        # Gradient step
        grad = compute_gradient(xk, A, b)
        x_grad = xk - step_size * grad
        
        # Proximal step using HJ-Prox for L1 norm
        x_prox, ls_iters = hj_prox(
            x_grad,
            t=step_size,
            f=l1_penalty,
            delta=delta_k,
            num_samples=num_samples,
            alpha=1
        )
        
        # Compute metrics
        fk = lasso_objective(x_prox, A, b, lambda_1)
        diff = torch.norm(x_prox - xk)
        f_hist.append(fk.item())
        diff_hist.append(diff.item())
        xk = x_prox.clone()
        
        if verbose and i % 10 == 0:
            print(f"PGD-HJ iter {i+1:4d}: f={fk.item():.6f}, ||Δx||={diff.item():.6e}, "
                  f"delta={delta_k:.6e}, time={time.time() - t0:.4f}s")
        
        if diff < tol:
            break
    
    return xk, torch.tensor(f_hist), torch.tensor(diff_hist), delta_k


# ============================================================================
# Algorithm 3: Proximal Point Method with HJ-Prox
# ============================================================================

def proximal_point_lasso(
    x0, A, b, lambda_1, gamma, max_iters=1000, 
    num_samples=100, tol=1e-6, verbose=True
):
    """
    Proximal Point Method for solving the LASSO problem:
      minimize 0.5 * ||Ax - b||^2 + lambda * ||x||_1
    
    Iteratively computes: x^(k+1) = prox_{γF}(x^k)
    where F(x) = 0.5||Ax - b||^2 + λ||x||_1 is the full objective.
    
    Uses delta = 150000/(k+1)^(2+EPS) annealing schedule.
    
    Parameters:
    -----------
    x0 : torch.Tensor
        Initial point
    A : torch.Tensor
        Design matrix
    b : torch.Tensor
        Observation vector
    lambda_1 : float
        L1 regularization parameter
    gamma : float
        Fixed proximal parameter (step size)
    max_iters : int
        Maximum number of iterations
    num_samples : int
        Number of samples for HJ-Prox
    tol : float
        Convergence tolerance
    verbose : bool
        Whether to print progress
        
    Returns:
    --------
    xk : torch.Tensor
        Final iterate
    f_hist : torch.Tensor
        Objective values
    diff_hist : torch.Tensor
        Iterate differences
    delta_k : float
        Final delta value
    """
    xk = x0.clone()
    f_hist = []
    diff_hist = []
    delta_k = None
    
    # Define full LASSO objective function for HJ-Prox (batch-aware)
    def lasso_objective_batch(x_batch):
        """
        Compute full LASSO objective for a batch of vectors.
        Args:
            x_batch: shape (batch_size, n)
        Returns:
            objectives: shape (batch_size,)
        """
        if x_batch.dim() == 1:
            x_batch = x_batch.unsqueeze(0)
        
        batch_size = x_batch.shape[0]
        
        # Smooth term: 0.5 * ||Ax - b||^2
        Ax = torch.matmul(x_batch, A.t())
        b_expanded = b.squeeze().unsqueeze(0).expand(batch_size, -1)
        residuals = Ax - b_expanded
        smooth_term = 0.5 * torch.sum(residuals ** 2, dim=1)
        
        # Non-smooth term: λ * ||x||_1
        l1_term = lambda_1 * torch.sum(torch.abs(x_batch), dim=1)
        
        return smooth_term + l1_term
    
    # Single objective for tracking
    def lasso_objective_single(x_vec):
        """Compute objective for a single vector."""
        x_flat = x_vec.squeeze()
        residual = A @ x_flat - b.squeeze()
        smooth = 0.5 * torch.sum(residual ** 2)
        l1 = lambda_1 * torch.sum(torch.abs(x_flat))
        return (smooth + l1).item()
    
    if verbose:
        initial_obj = lasso_objective_single(xk)
        print("Starting Proximal Point Method with HJ-Prox for LASSO...")
        print(f"λ = {lambda_1}, γ = {gamma}")
        print(f"Initial objective: {initial_obj:.6f}")
        print("-" * 70)
    
    for i in range(max_iters):
        t0 = time.time()
        
        # Compute delta with annealing schedule
        k = i + 1
        delta_k = 125000 / (k ** (2 + EPS))
        
        # Proximal Point step: x^(k+1) = prox_{γF}(x^k)
        x_prox, ls_iters = hj_prox(
            xk,
            t=gamma,
            f=lasso_objective_batch,
            delta=delta_k,
            num_samples=num_samples,
            alpha=1
        )
        
        # Compute metrics
        fk = lasso_objective_single(x_prox)
        diff = torch.norm(x_prox - xk)
        f_hist.append(fk)
        diff_hist.append(diff.item())
        
        if verbose and i % 10 == 0:
            print(f"PPM-HJ iter {i+1:4d}: f={fk:.6f}, ||Δx||={diff.item():.6e}, "
                  f"delta={delta_k:.6e}, ls_iters={ls_iters}, "
                  f"time={time.time() - t0:.4f}s")
        
        # Update iterate
        xk = x_prox.clone()
        
        # Check convergence
        if diff < tol:
            if verbose:
                print(f"\nPPM-HJ converged at iteration {i+1}")
                print(f"Final objective: {fk:.6f}")
            break
    
    return xk, torch.tensor(f_hist), torch.tensor(diff_hist), delta_k


print("✓ All algorithms and helper functions loaded successfully")

## Problem definition


In [ ]:
# ============================================================================
# CHUNK 2: DATA GENERATION
# ============================================================================

seed = 100
np.random.seed(seed)
torch.manual_seed(seed)

# Problem dimensions
dim = 500
A = torch.randn(int(dim/2), dim, device=device)

# Create a sparse x_true
x_true = torch.zeros(dim, 1, device=device)
x_true[400:410] = 1

# Generate b = A x_true + noise
noise_level = 0.1
noise = noise_level * torch.randn(int(dim/2), 1, device=device)
b = A @ x_true + noise

# Initial point
x0 = torch.zeros((dim, 1), dtype=torch.float32, device=device)

# Parameters
lambda_1 = 1

# Compute Lipschitz constant for step size
sigma_max = torch.linalg.norm(A, ord=2)
L = sigma_max**2
step_size = 1.0 / L

print("="*60)
print("Running LASSO Optimization Comparison")
print("="*60)
print(f"Problem size: A is {A.shape}, x is {x_true.shape}")
print(f"Regularization parameter λ = {lambda_1}")
print(f"Lipschitz constant L = {L:.4f}")
print(f"Step size for PGD = {step_size:.6f}")
print("="*60)

# Store results
results = {}

## Analytical PGD (closed-form prox)


In [ ]:
# ============================================================================
# CHUNK 3: RUN ALGORITHM 1 - Analytical PGD
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 1: Analytical PGD...")
print("="*60)

x_analytical, f_hist_analytical, diff_hist_analytical = proximal_gradient_descent_analytical(
    x0=x0, A=A, b=b, lambda_1=1,
    step_size=step_size*0.085,
    max_iters=10000,
    tol=1e-30,
    verbose=True
)
results['Analytical'] = (x_analytical, f_hist_analytical, diff_hist_analytical)

print(f"\n✓ Analytical PGD completed")
print(f"  - Converged in {len(f_hist_analytical)} iterations")
print(f"  - Final objective: {f_hist_analytical[-1]:.6f}")


## PGD with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 4: RUN ALGORITHM 2 - PGD with HJ-Prox
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 2: PGD with HJ-Prox...")
print("="*60)

x_hjprox, f_hist_hjprox, diff_hist_hjprox, final_delta = proximal_gradient_descent_lasso(
    x0=x0, A=A, b=b, lambda_1=1,
    step_size=step_size*0.085, #As mentioned in paper, we need small step sizes to help control errors
    max_iters=10000,
    num_samples=1000,
    delta=0.01,
    tol=1e-16,
    verbose=True
)
results['HJ-Prox'] = (x_hjprox, f_hist_hjprox, diff_hist_hjprox)

print(f"\n✓ PGD-HJ completed")
print(f"  - Converged in {len(f_hist_hjprox)} iterations")
print(f"  - Final objective: {f_hist_hjprox[-1]:.6f}")
print(f"  - Final delta: {final_delta:.6e}")

## Comparison: PGD vs PGD-HJ


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
# ============================================================================
# CHUNK 5: GENERATE FIGURES - PGD vs PGD-HJ Comparison
# ============================================================================

print("\n" + "="*60)
print("Generating figures: PGD vs PGD-HJ comparison...")
print("="*60)

# Prepare data
true_signal = x_true.detach().cpu().numpy().flatten()
analytical_signal = results['Analytical'][0].detach().cpu().numpy().flatten()
hjprox_signal = results['HJ-Prox'][0].detach().cpu().numpy().flatten()
analytical_hist = results['Analytical'][1]
hjprox_hist = results['HJ-Prox'][1]

start, end = 395, 415
idx = np.arange(start, end)

# --- Figure 1: Ground Truth ---
plt.figure(figsize=(11, 10))
plt.plot(idx, true_signal[idx], 'o', markersize=10,
         markerfacecolor='none', markeredgecolor='black',
         markeredgewidth=3)
plt.ylabel('Coefficient Value', fontsize=40)
plt.xlabel('Coefficients', fontsize=40)
plt.title('Ground Truth (395-415)', fontsize=40)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/lasso_ground_truth.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 2: HJ-Prox Solution ---
plt.figure(figsize=(11, 10))
plt.plot(idx, hjprox_signal[idx], 's', markersize=8, color='blue')
plt.ylabel('Coefficient Value', fontsize=40)
plt.xlabel('Coefficient Index', fontsize=40)
plt.title('PGD-HJ', fontsize=40)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/lasso_hjprox.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 3: Analytical PGD Solution ---
plt.figure(figsize=(11, 10))
plt.plot(idx, analytical_signal[idx], '^', markersize=8, color='red')
plt.ylabel('Coefficient Value', fontsize=40)
plt.xlabel('Coefficients', fontsize=40)
plt.title('PGD', fontsize=40)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/lasso_pgd.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 4: Objective Function Convergence (PGD vs PGD-HJ) ---
plt.figure(figsize=(11, 10))
plt.semilogy(hjprox_hist, '-', linewidth=3,
             label=f'PGD-HJ: {hjprox_hist[-1].item():.3f}')
plt.semilogy(analytical_hist, '--', linewidth=3,
             label=f'PGD: {analytical_hist[-1].item():.3f}')
plt.ylabel('Objective (log scale)', fontsize=40)
plt.xlabel('Iteration', fontsize=40)
plt.title('PGD Convergence', fontsize=40)
plt.legend(fontsize=40, loc='upper left')
plt.grid(True, which='both', alpha=0.3)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
max_iter = len(hjprox_hist)
tick_positions = np.arange(0, max_iter + 1, 2500)
plt.gca().set_xticks(tick_positions)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/lasso_objectives.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: lasso_ground_truth.pdf, lasso_hjprox.pdf, lasso_pgd.pdf, lasso_objectives.pdf")


## Proximal-point method with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 6: RUN ALGORITHM 3 - PPM with HJ-Prox (Lower Sample size to show difference)
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 3: Proximal Point Method with HJ-Prox...")
print("="*60)

x_ppm, f_ppm, diff_ppm, _ = proximal_point_lasso(
    x0, A, b, lambda_1=1, gamma= step_size*0.085,
    max_iters=10000,
    num_samples=500,
    tol=1e-6,
    verbose=True
)

print(f"\n✓ PPM-HJ completed")
print(f"  - Converged in {len(f_ppm)} iterations")
print(f"  - Final objective: {f_ppm[-1]:.6f}")

## PGD with HJ-Prox (smaller sample count)


In [ ]:
# ============================================================================
# CHUNK 7: RUN ALGORITHM 2 - PGD with HJ-Prox (Lower Sample size to show difference)
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 2: PGD with HJ-Prox...")
print("="*60)

x_hjprox, f_hist_hjprox, diff_hist_hjprox, final_delta = proximal_gradient_descent_lasso(
    x0=x0, A=A, b=b, lambda_1=1,
    step_size=step_size*0.085, #As mentioned in paper, we need small step sizes to help control errors
    max_iters=10000,
    num_samples=500,
    delta=0.01,
    tol=1e-16,
    verbose=True
)
results['HJ-Prox'] = (x_hjprox, f_hist_hjprox, diff_hist_hjprox)

print(f"\n✓ PGD-HJ completed")
print(f"  - Converged in {len(f_hist_hjprox)} iterations")
print(f"  - Final objective: {f_hist_hjprox[-1]:.6f}")
print(f"  - Final delta: {final_delta:.6e}")

## Comparison: PGD-HJ vs PPM-HJ


In [ ]:
# ============================================================================
# CHUNK 7: GENERATE FIGURES - PGD-HJ vs PPM-HJ Comparison
# ============================================================================

print("\n" + "="*60)
print("Generating figures: PGD-HJ vs PPM-HJ comparison...")
print("="*60)

# Prepare data
ppm_signal = x_ppm.detach().cpu().numpy().flatten()
ppm_hist = f_ppm

# --- Figure 5: PPM-HJ Solution ---
plt.figure(figsize=(11, 10))
plt.plot(idx, ppm_signal[idx], '^', markersize=8, color='red')
plt.ylabel('Coefficient Value', fontsize=40)
plt.xlabel('Coefficients', fontsize=40)
plt.title('PPM-HJ', fontsize=40)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/lasso_ppm.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 6: Objective Function Convergence (PGD-HJ vs PPM-HJ) ---
plt.figure(figsize=(11, 10))
plt.semilogy(hjprox_hist, '-', linewidth=3,
             label=f'PGD-HJ: {hjprox_hist[-1].item():.3f}')
plt.semilogy(ppm_hist, '--', linewidth=3,
             label=f'PPM-HJ: {ppm_hist[-1].item():.3f}')
plt.ylabel('Objective (log scale)', fontsize=40)
plt.xlabel('Iteration', fontsize=40)
plt.title('LASSO', fontsize=40)
plt.legend(fontsize=40, loc='upper left')
plt.grid(True, which='both', alpha=0.3)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
max_iter = len(hjprox_hist)
tick_positions = np.arange(0, max_iter + 1, 2500)
plt.gca().set_xticks(tick_positions)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/lasso_objectives_VS_PPM.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: lasso_ppm.pdf, lasso_objectives_VS_PPM.pdf")
print("\n" + "="*60)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print("="*60)